# broadcast-initial-weights — faded example 2: Broadcast across the full state_dict including buffers

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-initial-weights`. The last cell reports your progress on the `Distributed: broadcast initial weights` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: broadcast initial weights` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-initial-weights`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-initial-weights"
DD_SUBTOPIC = "Distributed: broadcast initial weights"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Parameters alone omit BatchNorm buffers (`running_mean`, `running_var`). Iterating `model.state_dict().values()` covers params AND buffers, so every replica becomes fully identical to rank 0. The order is deterministic across ranks since the module graphs match.

## Faded exercise 2

### Sync the whole state_dict

Complete `broadcast_state` so it broadcasts EVERY tensor in the model's state dict (params and BatchNorm buffers) from rank 0, returning the count of tensors broadcast. The fake collective and divergent models are provided.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
class FakeDist:
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Sequential(t.nn.Linear(3, 3, bias=False), t.nn.BatchNorm1d(3))
    with t.no_grad():
        m[0].weight.fill_(float(rank + 1))
        m[1].running_mean.fill_(float(10 * (rank + 1)))
    return m

def broadcast_state(model, fake):
    fake.reset()
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above

rank0 = build_model(0)
fake = FakeDist([v.clone() for v in rank0.state_dict().values()])
rank1 = build_model(1)
n = broadcast_state(rank1, fake)
print(n, rank1.state_dict()['1.running_mean'].tolist())


def _test():
    r0 = build_model(0)
    f = FakeDist([v.clone() for v in r0.state_dict().values()])
    r1 = build_model(1)
    n_params = sum(1 for _ in r1.parameters())
    n = broadcast_state(r1, f)
    assert n == len(r0.state_dict()), 'must broadcast every state_dict tensor'
    assert n > n_params, 'count must include buffers, not just params'
    assert t.equal(r1.state_dict()['1.running_mean'], r0.state_dict()['1.running_mean']), 'running_mean must sync'
    assert t.equal(r1.state_dict()['0.weight'], r0.state_dict()['0.weight']), 'weight must sync'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class FakeDist:
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Sequential(t.nn.Linear(3, 3, bias=False), t.nn.BatchNorm1d(3))
    with t.no_grad():
        m[0].weight.fill_(float(rank + 1))
        m[1].running_mean.fill_(float(10 * (rank + 1)))
    return m

def broadcast_state(model, fake):
    fake.reset()
    count = 0
    for tensor in model.state_dict().values():
        fake.broadcast(tensor, src=0)
        count += 1
    return count

rank0 = build_model(0)
fake = FakeDist([v.clone() for v in rank0.state_dict().values()])
rank1 = build_model(1)
n = broadcast_state(rank1, fake)
print(n, rank1.state_dict()['1.running_mean'].tolist())
```
</details>